# Data cleaning — 2025 HMDA Public LAR

Issue #1: prepare the notebook with the data cleaning script.

Source file: `2025_public_lar_csv.csv` (FFIEC HMDA public LAR, 2025), ~5.1 GB, 13,543,606 rows, 99 columns.
Not committed to git (see `.gitignore`) — place it at `data/2025_public_lar_csv.csv` in the repo root,
or point `DATA_PATH` below at wherever you keep it locally.

Missing values in the raw file are encoded as the literal string `"NA"` — with one confirmed exception:
`census_tract` contains at least one lowercase `"na"`. We treat both as null everywhere below.

**Why polars instead of pandas:** the full file is 5 GB / 13.5M rows / 99 columns. Loading that into
pandas can easily balloon to 3-5x the on-disk size in memory (every string cell becomes a Python object),
which risks running out of RAM on a typical laptop. [polars](https://pola.rs) uses a compact columnar
(Arrow) memory layout and a lazy/streaming execution model, so we can filter, derive columns, and
aggregate over the full file without ever holding all of it in memory at once.

## How to run this notebook

1. Download the raw file (see repo README) — you'll need ~5 GB of free disk space.
2. **Set `DATA_PATH` below to wherever you saved `2025_public_lar_csv.csv` locally.** If you placed
   it at `data/2025_public_lar_csv.csv` in the repo root (the convention in the README), the default
   already points there and you don't need to change anything.
3. Run all cells top to bottom (Kernel → Restart & Run All). Nothing else needs editing.
4. When it finishes, `data/processed/train.parquet`, `val.parquet`, and `test.parquet` are the clean,
   split datasets — load them with `pd.read_parquet(...)` or `pl.read_parquet(...)` in your own
   modeling notebook.

Because the split is deterministic (`RANDOM_STATE` fixed below), everyone who runs this against the
same raw file gets identical train/val/test splits — no need to pass the output files around.

In [1]:
import polars as pl
from pathlib import Path

DATA_PATH = Path("../data/2025_public_lar_csv.csv")  # <-- point this at YOUR local copy of the raw file
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # remainder (0.15) goes to test

# Columns that look numeric but are really identifiers/geocodes (leading zeros, no arithmetic
# ever done on them, and in practice contain stray non-numeric "NA"/"na" values) — read as strings.
ID_COLS_AS_STRING = ["lei", "derived_msa_md", "county_code", "census_tract"]

# Demographic fields used to check the loan decision was fair, kept in every split at proportional rates.
STRATA_COLS = ["derived_race", "derived_ethnicity", "derived_sex"]


def scan_lar(path=DATA_PATH) -> pl.LazyFrame:
    """Lazily scan the raw LAR CSV with consistent null handling and dtypes."""
    return pl.scan_csv(
        path,
        null_values=["NA", "na"],
        schema_overrides={c: pl.Utf8 for c in ID_COLS_AS_STRING},
    )

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Can't find the raw LAR file at {DATA_PATH.resolve()}.\n"
        "Edit DATA_PATH in the cell above to point at wherever you saved "
        "'2025_public_lar_csv.csv' on your machine."
    )
print(f"Using data file: {DATA_PATH.resolve()} ({DATA_PATH.stat().st_size / 1e9:.2f} GB)")

Using data file: C:\Users\Jessie\Desktop\HEC\ISAF\US_morgage_acceptance\data\2025_public_lar_csv.csv (5.11 GB)


## 1. Quick peek

Look at the schema and a handful of rows before running anything over the full file.

In [3]:
preview = scan_lar().head(5).collect()
preview.glimpse()

Rows: 5
Columns: 99
$ activity_year                            <i64> 2025, 2025, 2025, 2025, 2025
$ lei                                      <str> '549300QFJDBOCX7WKY08', '549300QFJDBOCX7WKY08', '549300QFJDBOCX7WKY08', '549300QFJDBOCX7WKY08', '549300QFJDBOCX7WKY08'
$ derived_msa_md                           <str> '23104', '99999', '23104', '99999', '99999'
$ state_code                               <str> 'TX', 'TX', 'TX', 'TX', 'TX'
$ county_code                              <str> '48439', '48467', '48439', '48467', '48133'
$ census_tract                             <str> '48439104505', '48467950200', '48439111027', '48467950400', '48133950202'
$ conforming_loan_limit                    <str> 'C', 'C', 'C', 'C', null
$ derived_loan_product_type                <str> 'Conventional:First Lien', 'Conventional:First Lien', 'Conventional:First Lien', 'Conventional:First Lien', 'Conventional:First Lien'
$ derived_dwelling_category                <str> 'Single Family (1-4 Units):Site-Built', '

## 2. EDA on the full file

These aggregations run in polars' streaming engine — a single pass over the file per cell, without
materializing all 13.5M rows in memory.

In [4]:
n_total = scan_lar().select(pl.len()).collect().item()
n_total

13543606

### 2a. `action_taken` distribution

Only `1` (originated), `2` (approved but not accepted), and `3` (denied) are clean, comparable credit
decisions. The rest — withdrawn, incomplete, purchased loan, preapproval-only — don't represent a
lender's accept/reject decision and are dropped.

In [5]:
action_counts = (
    scan_lar()
    .group_by("action_taken")
    .len()
    .sort("action_taken")
    .collect(engine="streaming")
    .with_columns((pl.col("len") / n_total * 100).round(2).alias("pct"))
)
action_counts

action_taken,len,pct
i64,u32,f64
1,6827891,50.41
2,417961,3.09
3,2115090,15.62
4,1710723,12.63
5,695006,5.13
6,1578064,11.65
7,50499,0.37
8,148372,1.1


### 2b. Missingness per column

Full-file null rates, worst offenders first. Column-level handling (drop / impute / encode) is left to
the team once we've agreed which predictors go into the model — this is diagnostic only.

In [6]:
null_counts = scan_lar().select(pl.all().null_count()).collect(engine="streaming")

missing_pct = (
    null_counts.transpose(include_header=True, header_name="column", column_names=["null_count"])
    .with_columns((pl.col("null_count") / n_total * 100).round(2).alias("pct_missing"))
    .filter(pl.col("pct_missing") > 0)
    .sort("pct_missing", descending=True)
)
missing_pct

column,null_count,pct_missing
str,u32,f64
"""applicant_ethnicity_4""",13543003,100.0
"""applicant_ethnicity_5""",13543276,100.0
"""co_applicant_ethnicity_4""",13543429,100.0
"""co_applicant_ethnicity_5""",13543520,100.0
"""co_applicant_race_5""",13542421,99.99
…,…,…
"""conforming_loan_limit""",56872,0.42
"""applicant_ethnicity_1""",4332,0.03
"""applicant_race_1""",2888,0.02


### 2c. Protected-attribute distributions

`derived_race`, `derived_ethnicity`, and `derived_sex` are HMDA's own simplified demographic fields —
these are what we'll stratify the train/val/test split on, and what the team's fairness metrics will
condition on later.

In [7]:
for col in STRATA_COLS:
    print(f"\n{col}:")
    print(
        scan_lar()
        .group_by(col)
        .len()
        .sort("len", descending=True)
        .collect(engine="streaming")
        .with_columns((pl.col("len") / n_total * 100).round(2).alias("pct"))
    )


derived_race:


shape: (9, 3)
┌─────────────────────────────────┬─────────┬───────┐
│ derived_race                    ┆ len     ┆ pct   │
│ ---                             ┆ ---     ┆ ---   │
│ str                             ┆ u32     ┆ f64   │
╞═════════════════════════════════╪═════════╪═══════╡
│ White                           ┆ 7674584 ┆ 56.67 │
│ Race Not Available              ┆ 3700244 ┆ 27.32 │
│ Black or African American       ┆ 1029097 ┆ 7.6   │
│ Asian                           ┆ 724554  ┆ 5.35  │
│ Joint                           ┆ 262328  ┆ 1.94  │
│ American Indian or Alaska Nati… ┆ 90834   ┆ 0.67  │
│ 2 or more minority races        ┆ 31302   ┆ 0.23  │
│ Native Hawaiian or Other Pacif… ┆ 27775   ┆ 0.21  │
│ Free Form Text Only             ┆ 2888    ┆ 0.02  │
└─────────────────────────────────┴─────────┴───────┘

derived_ethnicity:


shape: (5, 3)
┌─────────────────────────┬─────────┬───────┐
│ derived_ethnicity       ┆ len     ┆ pct   │
│ ---                     ┆ ---     ┆ ---   │
│ str                     ┆ u32     ┆ f64   │
╞═════════════════════════╪═════════╪═══════╡
│ Not Hispanic or Latino  ┆ 8279057 ┆ 61.13 │
│ Ethnicity Not Available ┆ 3542095 ┆ 26.15 │
│ Hispanic or Latino      ┆ 1415386 ┆ 10.45 │
│ Joint                   ┆ 302736  ┆ 2.24  │
│ Free Form Text Only     ┆ 4332    ┆ 0.03  │
└─────────────────────────┴─────────┴───────┘

derived_sex:


shape: (4, 3)
┌───────────────────┬─────────┬───────┐
│ derived_sex       ┆ len     ┆ pct   │
│ ---               ┆ ---     ┆ ---   │
│ str               ┆ u32     ┆ f64   │
╞═══════════════════╪═════════╪═══════╡
│ Male              ┆ 4173097 ┆ 30.81 │
│ Joint             ┆ 4065993 ┆ 30.02 │
│ Female            ┆ 2662580 ┆ 19.66 │
│ Sex Not Available ┆ 2641936 ┆ 19.51 │
└───────────────────┴─────────┴───────┘


## 3. Binary target

- `target = 1` → originated (`action_taken` in `{1, 2}`: loan originated, or approved but not
  accepted by the applicant — both are lender *approvals*)
- `target = 0` → denied (`action_taken == 3`)
- everything else is dropped (not a credit decision)

In [8]:
labeled = (
    scan_lar()
    .with_row_index("_row_id")  # stable id, carried through to `cleaned` for the split later
    .filter(pl.col("action_taken").is_in([1, 2, 3]))
    .with_columns(pl.col("action_taken").is_in([1, 2]).cast(pl.Int8).alias("target"))
)

target_rate = labeled.select(pl.col("target").mean()).collect().item()
n_labeled = labeled.select(pl.len()).collect().item()
print(f"{n_labeled:,} rows kept of {n_total:,} ({n_labeled / n_total:.1%})")
print(f"target=1 (originated) rate: {target_rate:.3f}")

9,360,942 rows kept of 13,543,606 (69.1%)
target=1 (originated) rate: 0.774


## 4. Cleaning decisions

Checked each candidate column/row issue against the actual data (conditioning on `target`, checking
cardinality, checking which values fail to parse) and against the
[official field documentation](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields)
rather than assuming — a few columns that looked like leakage on paper turned out not to be, and vice
versa. Column drops (4a-4d), duplicate removal (4e), and value standardization (4f) are all applied to
`labeled` before the final row-drop (4g) produces `cleaned`, the frame the split is built from.

### 4a. Leakage columns — drop

Confirmed by comparing each column's null/zero rate between denied and originated rows: these are only
populated *after* a decision was made, or directly re-encode the decision itself, so they'd trivially
leak the target if used as a naive predictor.

| column | % null when denied | % null when originated |
|---|---|---|
| `action_taken` | always `3` | always `1` or `2` — this is the raw field `target` was derived from, a perfect 1:1 leak, stronger than anything below |
| `denial_reason_1` | always has a value (a real reason code) | always `10` = "not applicable" |
| `interest_rate` | 98.4% | 0.03% |
| `rate_spread` | 98.5% | 9.5% |
| `purchaser_type` | 100% = "not applicable" (0) | 43% = "not applicable" (nonzero ⟹ originated with certainty) |
| `total_loan_costs` | 98.4% | 30.2% |
| `origination_charges` | 98.4% | 30.2% |
| `hoepa_status` | high-cost-mortgage flag set from pricing terms (rate spread vs. APOR) that, like `interest_rate`/`rate_spread` above, are only finalized for closed loans | — |
| `preapproval` | preapproved applications are pre-screened for creditworthiness before this record exists — approval correlates with having passed that screen, not with the features themselves | — |
| `discount_points`, `lender_credits` | weaker but still skewed (~98% null when denied vs. 63-74% null when originated, per §4e's original analysis) — initially standardized rather than dropped, revisited per team discussion | — |

Columns checked and **kept** despite initially looking suspicious, because the data shows near-equal
missingness regardless of the decision: `loan_term`, `aus_1..5`, `prepayment_penalty_term`,
`intro_rate_period`, `total_points_and_fees`.

In [9]:
LEAKAGE_COLS = [
    "action_taken", "denial_reason_1", "interest_rate", "rate_spread",
    "purchaser_type", "total_loan_costs", "origination_charges",
    "hoepa_status", "preapproval", "discount_points", "lender_credits",
]

### 4b. Redundant columns — drop

Cross-checked the [official HMDA LAR field documentation](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields)
against every kept column. These duplicate information already captured elsewhere:

- `applicant_age_above_62`, `co_applicant_age_above_62` — derivable from the age band already kept
  (`applicant_age`/`co_applicant_age`), give or take the 55-64 band straddling the 62 cutoff.
- `applicant_ethnicity_1`, `applicant_race_1`, `applicant_sex` and the co-applicant equivalents —
  `derived_ethnicity`/`derived_race`/`derived_sex` are HMDA's own standardized aggregation of these
  exact raw fields (including joint/multi-race handling), so keeping both is redundant.
- `applicant_ethnicity_observed`, `applicant_race_observed`, `applicant_sex_observed` and the
  co-applicant equivalents — records *how* the demographic data was collected (visual observation vs.
  self-report), not a demographic or loan attribute itself.

Worth a second look later: the `*_observed` fields specifically could be read as a fairness-mechanism
signal (whether the lender had visual demographic information before deciding) rather than pure
redundancy — noted per team discussion, dropping per the team's call.

In [10]:
REDUNDANT_COLS = [
    "applicant_age_above_62", "co_applicant_age_above_62",
    "applicant_ethnicity_1", "co_applicant_ethnicity_1",
    "applicant_race_1", "co_applicant_race_1",
    "applicant_sex", "co_applicant_sex",
    "applicant_ethnicity_observed", "co_applicant_ethnicity_observed",
    "applicant_race_observed", "co_applicant_race_observed",
    "applicant_sex_observed", "co_applicant_sex_observed",
]

### 4c. Constant columns — drop

Checked every kept column's cardinality: `activity_year` is the only column with exactly one distinct
value across all 9.36M labeled rows (this is a single-year file — every row is `2025`). Zero
information for modeling, so it's dropped. Nothing else came up constant — several columns are
*skewed* (e.g. `preapproval` is 97.8% "not requested"), but a skewed-not-constant column still has
hundreds of thousands of real rows on the minority side, which is a different situation from true
zero variance and not dropped here.

In [11]:
n_unique_per_col = labeled.select(pl.all().n_unique()).collect(engine="streaming")
CONSTANT_COLS = [
    c for c in n_unique_per_col.columns
    if n_unique_per_col[c][0] == 1 and c not in ("_row_id",)
]
print(f"Constant columns (dropped): {CONSTANT_COLS}")

Constant columns (dropped): ['activity_year']


### 4d. Near-empty columns — drop

Columns ≥90% missing *within the labeled data* (computed fresh here, not reused from the full-file
EDA above, since this decision should reflect the modeling-relevant subset). Mostly the multi-select
race/ethnicity overflow slots and secondary denial/AUS reasons — sparse because most applicants only
report one race/ethnicity, not because of a data quality problem, and several of these
(`denial_reason_2..4`, `aus_2..5`) are also leakage-adjacent (populated mainly for denied loans), so
the missingness and leakage concerns reinforce each other here. `derived_race`/`derived_ethnicity`
already carry the fairness-relevant signal, so dropping these loses little.

In [12]:
NEAR_EMPTY_THRESHOLD = 90.0  # percent missing

labeled_missing_pct = (
    labeled.select(pl.all().null_count()).collect(engine="streaming")
    .transpose(include_header=True, header_name="column", column_names=["null_count"])
    .with_columns((pl.col("null_count") / n_labeled * 100).round(2).alias("pct_missing"))
    .sort("pct_missing", descending=True)
)
NEAR_EMPTY_COLS = (
    labeled_missing_pct.filter(pl.col("pct_missing") >= NEAR_EMPTY_THRESHOLD)["column"].to_list()
)
print(f"{len(NEAR_EMPTY_COLS)} columns ≥{NEAR_EMPTY_THRESHOLD}% missing:")
labeled_missing_pct.filter(pl.col("pct_missing") >= NEAR_EMPTY_THRESHOLD)

26 columns ≥90.0% missing:


column,null_count,pct_missing
str,u32,f64
"""applicant_ethnicity_4""",9360484,100.0
"""applicant_ethnicity_5""",9360679,100.0
"""co_applicant_ethnicity_4""",9360804,100.0
"""co_applicant_ethnicity_5""",9360876,100.0
"""co_applicant_race_5""",9359993,99.99
…,…,…
"""denial_reason_2""",8894930,95.02
"""applicant_race_2""",8865576,94.71
"""aus_2""",8814646,94.16


### 4e. Duplicate rows — drop, keep first occurrence

Exact full-row duplicates (all 99 columns identical). HMDA's public file has no unique loan-level ID,
so a duplicate could be a genuine reporting artifact *or* two different loans that happen to report
identically on every disclosed field (common values in a small tract, rounded income/loan amounts) —
most duplicate groups are simple pairs, which reads as coincidence rather than error, and a few larger
clusters (up to 77 identical rows from one lender) look more like reporting artifacts. Per the team's
call, dropped either way: for each group of identical rows, keep the one that appears first in the raw
file (lowest `_row_id`) and drop the rest.

In [13]:
dup_hash = pl.struct(pl.all().exclude(["_row_id", "target"])).hash()
labeled = labeled.with_columns(dup_hash.alias("_dup_hash"))

is_first_occurrence = pl.col("_row_id") == pl.col("_row_id").min().over("_dup_hash")
n_before_dedup = labeled.select(pl.len()).collect().item()
labeled = labeled.filter(is_first_occurrence).drop("_dup_hash")
n_after_dedup = labeled.select(pl.len()).collect().item()

n_dropped_dupes = n_before_dedup - n_after_dedup
print(f"Dropped {n_dropped_dupes:,} duplicate rows ({n_dropped_dupes / n_before_dedup:.2%} of labeled rows), "
      f"kept first occurrence per group — {n_after_dedup:,} rows remain")

Dropped 16,767 duplicate rows (0.18% of labeled rows), kept first occurrence per group — 9,344,175 rows remain


### 4f. Value-level standardization

Several kept columns mix formats *within a single column*, discovered by checking which non-null
values fail to parse as numbers (and, for age, by noticing values that parse as numbers but are
obviously not real ages). Extended after cross-checking the
[official field documentation](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields)
column by column, which surfaced a second, different-looking sentinel we'd missed entirely.

- **6 columns use the literal string `"Exempt"`** in place of a number (`combined_loan_to_value_ratio`,
  `loan_term`, `intro_rate_period`, `property_value`, plus `discount_points`/`lender_credits` before
  they were dropped as leakage) — a real HMDA reporting exemption for smaller institutions, ~192-195K
  rows (~3%) each, 97.5% of which are exempt across all of them simultaneously (the same institutions,
  not independent gaps). Converted straight to null.
- **14 more columns use a *numeric* sentinel `1111`, also meaning "Exempt"** — a different encoding
  style than the string version above, which is why it wasn't caught the first pass: it parses as a
  valid number/category, it doesn't fail any "does this parse" check. Confirmed against the doc and the
  real data (~2.4-2.5% of rows in every one of them): `reverse_mortgage`, `open_end_line_of_credit`,
  `business_or_commercial_purpose`, `negative_amortization`, `interest_only_payment`, `balloon_payment`,
  `other_nonamortizing_features`, `manufactured_home_secured_property_type`,
  `manufactured_home_land_property_interest`, `applicant_credit_score_type`,
  `co_applicant_credit_score_type`, `submission_of_application`, `initially_payable_to_institution`,
  `aus_1`. Converted to null.
- **`debt_to_income_ratio`** mixes exact numbers (reported when DTI is 36-49%) with bucketed ranges at
  the extremes (`<20%`, `20%-<30%`, `30%-<36%`, `50%-60%`, `>60%`) plus the same `"Exempt"` pattern.
  Converted to a numeric estimate using each bucket's midpoint (`<20%`→10, `20%-<30%`→25, `30%-<36%`→33,
  `50%-60%`→55, `>60%`→70 — the last is a documented assumption, not a measured value).
- **`total_units`** is mostly an exact count but switches to a range for large multifamily properties
  (`5-24`, `25-49`, `50-99`, `100-149`, `>149`, 0.5% of rows) — same midpoint treatment
  (`>149`→200, also a documented assumption).
- **`applicant_age`** uses the sentinel `8888` ("age not available") in 2.8% of rows, mixed in with
  otherwise-consistent age-band strings (`25-34`, etc.). Converted to null.
- **`co_applicant_age`** uses `9999` in **58.5% of all rows** — not "unknown," this means *no
  co-applicant on the loan*, which is real, common, and plausibly fairness-relevant (solo vs. joint
  applications). Too large and too meaningful to silently null, so it's split into a `has_co_applicant`
  boolean (kept) before the sentinel values (`9999` and the rarer `8888`, "co-applicant exists but age
  unavailable") are nulled out of the age column itself.
- **Three fields have a *field-specific* "Not applicable" code, separate from the `1111` sentinel above**
  — checked each against its actual prevalence before deciding, since the same label means different
  things in different fields:
  - `applicant_credit_score_type` code `9` (11.5% of rows) and `co_applicant_credit_score_type` code `9`
    (21.0%) → converted to null, per team decision.
  - `co_applicant_credit_score_type` code `10` ("No co-applicant", 57.0%) → converted to null; the fact
    itself is already captured by `has_co_applicant`, so a credit-score-type value doesn't apply.
  - `initially_payable_to_institution` code `3` (2.9%) → converted to null, per team decision.
  - `loan_purpose` code `5` (0.05%, a few thousand rows) → converted to null.
  - **Deliberately NOT touched:** `manufactured_home_secured_property_type` code `3` and
    `manufactured_home_land_property_interest` code `5`, both also labeled "Not applicable" — but at
    **92%+ of all rows**, this is genuine skip logic (the correct answer for every non-manufactured-home
    loan), not missing data. Nulling these would have destroyed real signal in both columns.

Done here, once, rather than left for each teammate to rediscover and handle inconsistently across
different models — but note the midpoint choices above (`>60%`→70, `>149`→200) are approximations
baked into the shared data, not measured values; worth keeping in mind if precision at the extremes
matters for a particular analysis.

In [14]:
EXEMPT_ONLY_COLS = [
    "combined_loan_to_value_ratio", "loan_term", "intro_rate_period", "property_value",
]

SENTINEL_1111_COLS = [
    "reverse_mortgage", "open_end_line_of_credit", "business_or_commercial_purpose",
    "negative_amortization", "interest_only_payment", "balloon_payment",
    "other_nonamortizing_features", "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest", "submission_of_application", "aus_1",
]

dti = pl.col("debt_to_income_ratio")
dti_numeric = (
    pl.when(dti == "<20%").then(10.0)
    .when(dti == "20%-<30%").then(25.0)
    .when(dti == "30%-<36%").then(33.0)
    .when(dti == "50%-60%").then(55.0)
    .when(dti == ">60%").then(70.0)
    .when(dti == "Exempt").then(None)
    .otherwise(dti.cast(pl.Float64, strict=False))
)

tu = pl.col("total_units")
tu_numeric = (
    pl.when(tu == "5-24").then(14.5)
    .when(tu == "25-49").then(37.0)
    .when(tu == "50-99").then(74.5)
    .when(tu == "100-149").then(124.5)
    .when(tu == ">149").then(200.0)
    .otherwise(tu.cast(pl.Float64, strict=False))
)

labeled = labeled.with_columns(
    [pl.col(c).cast(pl.Float64, strict=False) for c in EXEMPT_ONLY_COLS]
    + [pl.when(pl.col(c) == 1111).then(None).otherwise(pl.col(c)).alias(c) for c in SENTINEL_1111_COLS]
    + [
        dti_numeric.alias("debt_to_income_ratio"),
        tu_numeric.alias("total_units"),
        (pl.col("co_applicant_age") != "9999").alias("has_co_applicant"),
        pl.when(pl.col("applicant_age") == "8888").then(None)
          .otherwise(pl.col("applicant_age")).alias("applicant_age"),
        pl.when(pl.col("co_applicant_age").is_in(["9999", "8888"])).then(None)
          .otherwise(pl.col("co_applicant_age")).alias("co_applicant_age"),
        pl.when(pl.col("applicant_credit_score_type").is_in([1111, 9])).then(None)
          .otherwise(pl.col("applicant_credit_score_type")).alias("applicant_credit_score_type"),
        pl.when(pl.col("co_applicant_credit_score_type").is_in([1111, 9, 10])).then(None)
          .otherwise(pl.col("co_applicant_credit_score_type")).alias("co_applicant_credit_score_type"),
        pl.when(pl.col("initially_payable_to_institution").is_in([1111, 3])).then(None)
          .otherwise(pl.col("initially_payable_to_institution")).alias("initially_payable_to_institution"),
        pl.when(pl.col("loan_purpose") == 5).then(None)
          .otherwise(pl.col("loan_purpose")).alias("loan_purpose"),
    ]
)
STANDARDIZED_COLS = (
    EXEMPT_ONLY_COLS + SENTINEL_1111_COLS
    + ["debt_to_income_ratio", "total_units", "applicant_age", "co_applicant_age",
       "applicant_credit_score_type", "co_applicant_credit_score_type",
       "initially_payable_to_institution", "loan_purpose"]
)
print("standardized:", STANDARDIZED_COLS)
print("added: has_co_applicant")

standardized: ['combined_loan_to_value_ratio', 'loan_term', 'intro_rate_period', 'property_value', 'reverse_mortgage', 'open_end_line_of_credit', 'business_or_commercial_purpose', 'negative_amortization', 'interest_only_payment', 'balloon_payment', 'other_nonamortizing_features', 'manufactured_home_secured_property_type', 'manufactured_home_land_property_interest', 'submission_of_application', 'aus_1', 'debt_to_income_ratio', 'total_units', 'applicant_age', 'co_applicant_age', 'applicant_credit_score_type', 'co_applicant_credit_score_type', 'initially_payable_to_institution', 'loan_purpose']
added: has_co_applicant


### 4g. Rows missing all core fields — drop

Rows missing **every one** of `income`, `property_value`, `combined_loan_to_value_ratio`, and
`debt_to_income_ratio` carry essentially no usable financial information regardless of what else is
filled in — a conservative drop (rows missing *some* but not all of these are left for the modeling
team to impute or handle).

In [15]:
CORE_FIELDS = ["income", "property_value", "combined_loan_to_value_ratio", "debt_to_income_ratio"]

missing_all_core = pl.all_horizontal([pl.col(c).is_null() for c in CORE_FIELDS])
n_dropped_rows = labeled.filter(missing_all_core).select(pl.len()).collect().item()
print(f"Dropping {n_dropped_rows:,} rows ({n_dropped_rows / n_after_dedup:.2%}) missing all of {CORE_FIELDS}")

cleaned = labeled.filter(~missing_all_core).drop(LEAKAGE_COLS + REDUNDANT_COLS + CONSTANT_COLS + NEAR_EMPTY_COLS)
n_cleaned = cleaned.select(pl.len()).collect().item()
n_cols_cleaned = len(cleaned.collect_schema())
print(f"cleaned: {n_cleaned:,} rows, {n_cols_cleaned} columns "
      f"(dropped {len(LEAKAGE_COLS)} leakage + {len(REDUNDANT_COLS)} redundant + "
      f"{len(CONSTANT_COLS)} constant + {len(NEAR_EMPTY_COLS)} near-empty columns)")

Dropping 211,219 rows (2.26%) missing all of ['income', 'property_value', 'combined_loan_to_value_ratio', 'debt_to_income_ratio']


cleaned: 9,132,956 rows, 50 columns (dropped 11 leakage + 14 redundant + 1 constant + 26 near-empty columns)


### 4h. Cleaning summary

Everything from section 4 in one place — what's excluded and why, what's standardized and how, and the
final feature list. Read this instead of scrolling back through the individual cells above.

In [16]:
STANDARDIZED_TO_MIDPOINT = ["debt_to_income_ratio", "total_units"]
STANDARDIZED_TO_NULL = [c for c in STANDARDIZED_COLS if c not in STANDARDIZED_TO_MIDPOINT]
FINAL_FEATURES = sorted(c for c in cleaned.collect_schema().names() if c != "_row_id")

print(f"FINAL FEATURES ({len(FINAL_FEATURES)}, includes target):")
print(FINAL_FEATURES)

print(f"\nEXCLUDED — leakage ({len(LEAKAGE_COLS)}):")
print(LEAKAGE_COLS)

print(f"\nEXCLUDED — redundant with another kept column ({len(REDUNDANT_COLS)}):")
print(REDUNDANT_COLS)

print(f"\nEXCLUDED — constant, zero information ({len(CONSTANT_COLS)}):")
print(CONSTANT_COLS)

print(f"\nEXCLUDED — ≥{NEAR_EMPTY_THRESHOLD}% missing ({len(NEAR_EMPTY_COLS)}):")
print(NEAR_EMPTY_COLS)

print(f"\nSTANDARDIZED — sentinel/placeholder values (e.g. \"Exempt\", 1111, 9999, 8888) converted to null ({len(STANDARDIZED_TO_NULL)}):")
print(STANDARDIZED_TO_NULL)

print(f"\nSTANDARDIZED — bucketed ranges converted to a numeric midpoint estimate ({len(STANDARDIZED_TO_MIDPOINT)}):")
print(STANDARDIZED_TO_MIDPOINT)

print("\nDERIVED — new columns added:")
print("  target            binary label, derived from action_taken (see section 3)")
print("  has_co_applicant  boolean, derived from co_applicant_age's sentinel value (see section 4f)")

print(f"\nROWS — dropped {n_dropped_dupes:,} exact duplicates ({n_dropped_dupes / n_before_dedup:.2%} of labeled rows), kept first occurrence per group")
print(f"ROWS — dropped {n_dropped_rows:,} ({n_dropped_rows / n_after_dedup:.2%}) missing all of {CORE_FIELDS}")
print(f"\n{n_total:,} raw rows → {n_labeled:,} labeled → {n_cleaned:,} cleaned rows, "
      f"{len(FINAL_FEATURES) - 1} features + target ({len(FINAL_FEATURES)} columns total)")

FINAL FEATURES (49, includes target):
['applicant_age', 'applicant_credit_score_type', 'aus_1', 'balloon_payment', 'business_or_commercial_purpose', 'census_tract', 'co_applicant_age', 'co_applicant_credit_score_type', 'combined_loan_to_value_ratio', 'conforming_loan_limit', 'construction_method', 'county_code', 'debt_to_income_ratio', 'derived_dwelling_category', 'derived_ethnicity', 'derived_loan_product_type', 'derived_msa_md', 'derived_race', 'derived_sex', 'ffiec_msa_md_median_family_income', 'has_co_applicant', 'income', 'initially_payable_to_institution', 'interest_only_payment', 'intro_rate_period', 'lei', 'lien_status', 'loan_amount', 'loan_purpose', 'loan_term', 'loan_type', 'manufactured_home_land_property_interest', 'manufactured_home_secured_property_type', 'negative_amortization', 'occupancy_type', 'open_end_line_of_credit', 'other_nonamortizing_features', 'property_value', 'reverse_mortgage', 'state_code', 'submission_of_application', 'target', 'total_units', 'tract_medi

## 5. Automated EDA report

The cells above cover what matters for the cleaning decisions; for a fuller look at every column's
distribution, correlations, and data-quality alerts, generate a standalone
[ydata-profiling](https://github.com/ydataai/ydata-profiling) report instead of cramming dozens more
plots into this notebook. It doesn't scale to 9M+ rows, so it runs on a random sample — a lazy,
streaming-safe sample (never materializes the full dataset), not `df.sample()` on an already-collected
frame. Saved as a separate HTML file rather than embedded, since these reports run large.

In [17]:
EDA_SAMPLE_N = 200_000
REPORT_PATH = OUT_DIR.parent.parent / "reports" / "eda_profile.html"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

sample_frac = EDA_SAMPLE_N / n_cleaned
eda_sample = (
    cleaned
    .with_columns((pl.int_range(pl.len()).shuffle(seed=RANDOM_STATE) / pl.len()).alias("_u"))
    .filter(pl.col("_u") < sample_frac)
    .drop("_u")
    .collect(engine="streaming")
    .to_pandas()
)
print(f"profiling sample: {eda_sample.shape}")

profiling sample: (200000, 50)


In [18]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    eda_sample,
    title="HMDA LAR 2025 — cleaned data profile (sample)",
    minimal=True,  # full mode's pairwise correlations don't scale well past ~50 columns
)
profile.to_file(REPORT_PATH)
print(f"wrote {REPORT_PATH}")

C:\Users\Jessie\AppData\Local\Temp\ipykernel_19888\2309104837.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:00<00:22,  2.18it/s]

  4%|▍         | 2/50 [00:00<00:15,  3.08it/s]

...

100%|██████████| 50/50 [00:01<00:00, 33.46it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

wrote ..\reports\eda_profile.html


## 6. Train / validation / test split

70/15/15, stratified on `target` **and** on `derived_race` / `derived_ethnicity` / `derived_sex`
jointly, so every demographic subgroup's approval rate — and its subgroup size — is preserved
proportionally across splits. This matters because the team's fairness metrics will be computed
per subgroup; a split that happened to under-represent a small subgroup in `val`/`test` would make
those metrics noisy or misleading.

Single cross-sectional year of data, so there's no temporal leakage risk to worry about.

**Memory-safe two-pass approach**, needed because we can't hold the full `cleaned` dataset (still
9M+ rows) in memory at once:

1. **Pass 1** — select only the id, target, and 3 strata columns from `cleaned`, assign each row a
   `split` label, keep just `(row id, target, split)`. This intermediate table is tiny (well under
   100 MB) regardless of file size.
2. **Pass 2** — re-run the `cleaned` lazy pipeline three times (once per split), join against the
   tiny assignment table from pass 1, and stream matching rows straight to a parquet file on disk.
   The full cleaned dataset is never materialized in memory all at once.

Within each stratum, the split label comes from a seeded random permutation of that stratum's rows
(`RANDOM_STATE` fixed for reproducibility) — proportions land almost exactly on 70/15/15 per subgroup
even though we never compute an alignment across strata sizes.

In [19]:
assignments = (
    cleaned
    .select(["_row_id", "target"] + STRATA_COLS)
    .with_columns(
        pl.concat_str([pl.col("target").cast(pl.Utf8)] + STRATA_COLS, separator="_").alias("_strata")
    )
    .with_columns([
        pl.int_range(pl.len()).shuffle(seed=RANDOM_STATE).over("_strata").alias("_rank"),
        pl.len().over("_strata").alias("_strata_size"),
    ])
)

frac = pl.col("_rank") / pl.col("_strata_size")
assignments = assignments.with_columns(
    pl.when(frac < TRAIN_FRAC).then(pl.lit("train"))
    .when(frac < TRAIN_FRAC + VAL_FRAC).then(pl.lit("val"))
    .otherwise(pl.lit("test"))
    .alias("split")
).select(["_row_id", "split"])

assignments = assignments.collect(engine="streaming")
print(f"assignment table: {assignments.height:,} rows, {assignments.estimated_size('mb'):.1f} MB in memory")
assignments.group_by("split").len().sort("split")

assignment table: 9,132,956 rows, 74.5 MB in memory


split,len
str,u32
"""test""",1369785
"""train""",6393228
"""val""",1369943


In [20]:
for name in ["train", "val", "test"]:
    ids = assignments.filter(pl.col("split") == name).select("_row_id")
    split_lf = cleaned.join(ids.lazy(), on="_row_id", how="inner").drop("_row_id")
    out_path = OUT_DIR / f"{name}.parquet"
    split_lf.sink_parquet(out_path)
    print(f"{name}: wrote {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")

train: wrote ..\data\processed\train.parquet (188.4 MB)


val: wrote ..\data\processed\val.parquet (42.0 MB)


test: wrote ..\data\processed\test.parquet (42.0 MB)


## 7. Sanity check

Confirm row counts, overall target rate, and — most importantly — that the target rate within each
demographic subgroup is consistent across splits (i.e. the stratification actually worked).

In [21]:
splits = {name: pl.read_parquet(OUT_DIR / f"{name}.parquet") for name in ["train", "val", "test"]}

for name, split_df in splits.items():
    print(f"{name}: {split_df.height:,} rows, target=1 rate = {split_df['target'].mean():.3f}")

train: 6,393,228 rows, target=1 rate = 0.775
val: 1,369,943 rows, target=1 rate = 0.775
test: 1,369,785 rows, target=1 rate = 0.775


In [22]:
subgroup_check = pl.concat(
    [
        split_df.group_by(STRATA_COLS)
        .agg(pl.len().alias("n"), pl.col("target").mean().alias("target_rate"))
        .with_columns(pl.lit(name).alias("split"))
        for name, split_df in splits.items()
    ]
).sort(STRATA_COLS + ["split"])

subgroup_check.head(20)

derived_race,derived_ethnicity,derived_sex,n,target_rate,split
str,str,str,u32,f64,str
"""2 or more minority races""","""Ethnicity Not Available""","""Female""",73,0.452055,"""test"""
"""2 or more minority races""","""Ethnicity Not Available""","""Female""",343,0.44898,"""train"""
"""2 or more minority races""","""Ethnicity Not Available""","""Female""",73,0.452055,"""val"""
"""2 or more minority races""","""Ethnicity Not Available""","""Joint""",38,0.657895,"""test"""
"""2 or more minority races""","""Ethnicity Not Available""","""Joint""",180,0.65,"""train"""
…,…,…,…,…,…
"""2 or more minority races""","""Free Form Text Only""","""Joint""",1,0.0,"""val"""
"""2 or more minority races""","""Free Form Text Only""","""Male""",8,0.375,"""train"""
"""2 or more minority races""","""Free Form Text Only""","""Male""",2,0.5,"""val"""


## 8. Handoff to the team

- Splits are written to `data/processed/{train,val,test}.parquet` (gitignored — too large for git,
  same as the raw file).
- Re-running this notebook against the same raw file with the same `RANDOM_STATE` reproduces byte-identical
  splits, so the notebook itself — not the parquet files — is the source of truth to share.
- Load with either `pl.read_parquet("data/processed/train.parquet")` or
  `pd.read_parquet("data/processed/train.parquet")`.
- Leakage columns, redundant columns (duplicated by `derived_race`/`derived_ethnicity`/`derived_sex` or
  the age bands, per the [official field docs](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields)),
  the constant `activity_year` column, and near-empty (≥90% missing) columns are already dropped
  (section 4) — see `LEAKAGE_COLS`, `REDUNDANT_COLS`, `CONSTANT_COLS`, and `NEAR_EMPTY_COLS` above for
  exactly what and why. Rows missing every core financial field are also already dropped.
- Value-level standardization (section 4f) is also already applied: `"Exempt"` strings, the numeric
  `1111` "Exempt" sentinel, sentinel age/credit-score codes, and bucketed ranges are all converted to
  proper numeric/null values — these columns are safe to use directly without each teammate re-deriving
  the same parsing logic. New column: `has_co_applicant` (boolean). Everything else (scaling, encoding,
  imputation strategy, feature selection) is left for the modeling team.
- A fuller automated EDA report (all columns, on a sample) is at `reports/eda_profile.html`
  (regenerate it the same way — also gitignored).